In [10]:
import json 
from pathlib import Path
import tiktoken
import pandas as pd

In [2]:
## intialize output directory
output_dir = Path("../data/chunked")

In [3]:
## intialize token length and overlap
chunk_size = 512
overlap = 0

tokenizer = tiktoken.get_encoding("cl100k_base")

# Function to chunk text into specified token length with overlap
def chunk_text(text):
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
    return chunks

In [4]:
## Load combined data
with open(r"..\data\combined_10k_data.json", "r") as f:
    data = json.load(f)

In [5]:
# Create chunks for each section and store in a list
all_chunks = []

print("Creating chunks...")
for company in data:
    file_name = company.get("file_name", "Unknown")
    # Extract company name from file_name (e.g., "aapl-20230930" -> "aapl")
    company_name = file_name.split("-")[0].upper() if file_name else "Unknown"
    
    # Iterate through sections
    for section in ["risk_factors", "management_discussion_and_analysis"]:
        text = company.get(section, "")
        
        if text:  # Only process if section exists and has content
            chunks = chunk_text(text)
            
            # Create chunk objects with metadata
            for chunk_id, chunk_content in enumerate(chunks):
                chunk_obj = {
                    "company_name": company_name,
                    "section": section,
                    "chunk_id": chunk_id,
                    "text": chunk_content
                }
                all_chunks.append(chunk_obj)

print(f"Total chunks created: {len(all_chunks)}\n")

Creating chunks...
Total chunks created: 610



In [9]:
with open(str(output_dir) + "/chunks_type1.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"Chunks saved to: {str(output_dir) + '/chunks_type1.json'}")

Chunks saved to: ..\data\chunked/chunks_type1.json


In [12]:
# chunk statistics
chunks_df = pd.DataFrame(all_chunks)

print("\n" + "="*50)
print("CHUNKS SUMMARY")
print("="*50)
print(f"Total chunks: {len(chunks_df)}")

print(f"\nChunks per company:")
print(chunks_df.groupby('company_name').size())

print(f"\nChunks per section:")
print(chunks_df.groupby('section').size())

print(f"\nChunks per company and section:")
print(chunks_df.groupby(['company_name', 'section']).size())

print(f"\n" + "="*50)
print("SAMPLE CHUNKS")
print("="*50)
for company in chunks_df['company_name'].unique()[:2]:
    for section in chunks_df['section'].unique()[:1]:
        sample = chunks_df[(chunks_df['company_name'] == company) & (chunks_df['section'] == section)].iloc[0]
        print(f"\n{company} - {section} (chunk {sample['chunk_id']}):")
        print(f"Text preview: {sample['text'][:200]}...")



CHUNKS SUMMARY
Total chunks: 610

Chunks per company:
company_name
AAPL          30
COCOCOLA     151
GOOG          32
LLY           45
MICROSOFT     45
NFLX          39
NVDA          51
ORACLE       105
UBER         112
dtype: int64

Chunks per section:
section
management_discussion_and_analysis    308
risk_factors                          302
dtype: int64

Chunks per company and section:
company_name  section                           
AAPL          management_discussion_and_analysis      7
              risk_factors                           23
COCOCOLA      management_discussion_and_analysis    103
              risk_factors                           48
GOOG          management_discussion_and_analysis      7
              risk_factors                           25
LLY           management_discussion_and_analysis     27
              risk_factors                           18
MICROSOFT     management_discussion_and_analysis     22
              risk_factors                           2